In [6]:
#Q1 - a

import numpy as np

# Load data
D = np.genfromtxt("lines.csv", delimiter=",", skip_header=1)

x1 = D[:, 0]
y1 = D[:, 3]

# Stack points
points = np.column_stack((x1, y1))

# Compute centroid
centroid = np.mean(points, axis=0)

# Subtract mean
Xc = points - centroid

# SVD
U, S, Vt = np.linalg.svd(Xc)

# Line direction
direction = Vt[0]

# Line normal
normal = Vt[1]

# Line equation: ax + by + c = 0
a, b = normal
c = -a*centroid[0] - b*centroid[1]

print("TLS Line parameters:", a, b, c)

TLS Line parameters: 0.7735616496467872 -0.6337210539312553 -3.794192210845812


In [7]:
#Q1 - b

import random

X_cols = D[:, :3]
Y_cols = D[:, 3:]

X_all = X_cols.flatten()
Y_all = Y_cols.flatten()

points = np.column_stack((X_all, Y_all))

def fit_line(p1, p2):
    a = p2[1] - p1[1]
    b = p1[0] - p2[0]
    c = -(a*p1[0] + b*p1[1])
    return a, b, c

def distance(a, b, c, point):
    x, y = point
    return abs(a*x + b*y + c) / np.sqrt(a*a + b*b)

def ransac(points, iterations=1000, threshold=0.5):
    best_inliers = []

    for _ in range(iterations):
        p1, p2 = random.sample(list(points), 2)
        a, b, c = fit_line(p1, p2)

        inliers = []
        for p in points:
            if distance(a, b, c, p) < threshold:
                inliers.append(p)

        if len(inliers) > len(best_inliers):
            best_inliers = inliers

    return np.array(best_inliers)

# Find 3 lines
remaining = points.copy()
lines = []

for i in range(3):
    inliers = ransac(remaining)
    lines.append(inliers)

    # remove inliers
    remaining = np.array([p for p in remaining if not any((p == inliers).all(1))])

In [1]:
#Q2 - Part 1: Click two points

import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib tk

img = cv2.imread("earrings.jpg")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

points = []

fig, ax = plt.subplots()
ax.imshow(img_rgb)
ax.set_title("Click two points on the earring")

def onclick(event):
    if event.xdata is not None and len(points) < 2:
        points.append((event.xdata, event.ydata))
        ax.plot(event.xdata, event.ydata, 'ro', markersize=5)
        fig.canvas.draw()
        if len(points) == 2:
            plt.close()
            print("Done! Points collected:", points)

fig.canvas.mpl_connect('button_press_event', onclick)
plt.show(block=True)

2026-04-21 16:31:35.655 python[2206:76416544] +[IMKClient subclass]: chose IMKClient_Legacy
2026-04-21 16:31:35.655 python[2206:76416544] +[IMKInputSession subclass]: chose IMKInputSession_Legacy


Done! Points collected: [(np.float64(98.6861471861472), np.float64(485.4567099567098)), (np.float64(481.02380952380963), np.float64(496.5389610389609))]


In [2]:
#Q2 - Part 2: Compute pixel length

p1, p2 = points
pixel_length = np.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)
print("Pixel length:", pixel_length)


Pixel length: 382.4982409513519
